In [ ]:
import time

import pandas as pd
import numpy as np

from nba_api.stats.endpoints import leaguedashplayerstats, commonteamroster, leaguedashplayerbiostats
from nba_api.stats.static import teams

from typing import Tuple

from sklearn.model_selection import train_test_split

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio

# Get raw data

In [ ]:
def get_player_stats(first_season: int, last_season: int) -> pd.DataFrame:
    assert first_season <= last_season, 'Last season must be later than first season'

    # Initialize a list to append data for each season
    list_seasons_data = []

    for season in range(first_season, last_season + 1):
        print(f"Fetching {season}...")

        # Initialize list to append data for each type of measure
        list_measure_data = []

        for measure_type in ['Base']:#, 'Advanced', 'Scoring']:
            # Call API
            stats = leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star='Regular Season',
                measure_type_detailed_defense=measure_type
            )

            # Store the data
            df_stats = stats.get_data_frames()[0]

            # Drop _RANK columns
            # since they just represent position by different criteria
            for col in df_stats.columns:
                if "RANK" in col:
                    df_stats.drop(col, axis=1, inplace=True)

            # Drop the MIN column in Advanced and Scoring
            # because they are minutes per game instead of total minutes
            if measure_type != 'Base':
                df_stats.drop('MIN', axis=1, inplace=True)

            # Append data to list of dataframes for the season
            list_measure_data.append(df_stats)

            # Sleep to not overload API calls
            time.sleep(0.5)
        
        # Merge the data for the season
        df_season = list_measure_data[0]
        for df_extra in list_measure_data[1:]:
            df_season = df_season.merge(df_extra)

        # Add SEASON column to identify data when we concatenate all seasons together
        df_season['SEASON'] = season

        # Append data to the list of dataframes for all seasons
        list_seasons_data.append(df_season)

    # Concatenate the data together at set an ID for each player + season combination    
    df = pd.concat(list_seasons_data, ignore_index=True)
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    return df

In [ ]:
def get_player_positions(first_season: int, last_season: int) -> pd.DataFrame:
    all_teams = teams.get_teams()
    rosters = []

    for team in all_teams:
        for season in range(first_season, last_season + 1):
            roster = commonteamroster.CommonTeamRoster(
                team_id=team['id'],
                season=season
            )
            df_roster = roster.get_data_frames()[0]
            df_roster['SEASON'] = season

            rosters.append(roster.get_data_frames()[0])
            time.sleep(0.5)  # avoid rate limiting

    df = pd.concat(rosters)[['PLAYER_ID', 'SEASON', 'POSITION']]
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    return df[['ID', 'POSITION']]

In [ ]:
def get_player_bio(first_season: int, last_season: int) -> pd.DataFrame:
    assert first_season <= last_season, 'Last season must be later than first season'

    # Initialize a list to append data for each season
    list_seasons_data = []

    for season in range(first_season, last_season + 1):
        print(f"Fetching {season}...")

        # Call API
        bio = leaguedashplayerbiostats.LeagueDashPlayerBioStats(season=season)

        # Store the data
        df_season = bio.get_data_frames()[0][['PLAYER_ID', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT']]

        # Sleep to not overload API calls
        time.sleep(0.5)

        # Add SEASON column to identify data when we concatenate all seasons together
        df_season['SEASON'] = season

        # Append data to the list of dataframes for all seasons
        list_seasons_data.append(df_season)

    # Concatenate the data together at set an ID for each player + season combination    
    df = pd.concat(list_seasons_data, ignore_index=True)
    df['ID'] = df['PLAYER_ID'].astype(str) + '_' + df['SEASON'].astype(str)

    # Rename columns
    df.rename(
        columns={'PLAYER_HEIGHT_INCHES': 'HEIGHT', 'PLAYER_WEIGHT': 'WEIGHT'},
        inplace=True
    )

    return df[['ID', 'HEIGHT', 'WEIGHT']]

In [ ]:
df_stats_raw = pd.read_csv("../assets/nba_player_stats.csv")
# df_stats_raw = get_player_stats(first_season=2016, last_season=2025)

In [ ]:
df_positions_raw = pd.read_csv("../assets/nba_player_positions.csv")
# df_positions_raw = get_player_positions(first_season=2016, last_season=2025)

In [ ]:
df_bio_raw = pd.read_csv("../assets/nba_player_bio.csv")
# df_bio_raw = get_player_bio(first_season=2016, last_season=2025)

In [ ]:
df_raw = df_stats_raw.merge(df_positions_raw, how='left').merge(df_bio_raw, how='left')

# Data cleaning

## Row treatment

Basketball statistics are highly dependent on sample size. When a player only plays 15 total minutes across a whole season, their advanced rates and percentages explode into unrealistic, hyper-volatile extremes.

We choose 250 minutes (equivalent to 10 games with 25 minutes of play)

In [ ]:
df_raw = df_raw[df_raw['MIN'] >= 250]

## Columns treatment

### Drop Unneeded

In [ ]:
df_raw = df_raw.drop(columns=['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'SEASON', 'TEAM_COUNT', 'GP', 'W', 'L', 'W_PCT'])

This leaves only `ID` and `POSITION` as string columns. All the remaining ones are numeric.

### Normalize (per 36 minutes)

Totals suffer from volume bias if we don't normalize them. We follow the standard of giving stats per 36 minutes.

In [ ]:
for col in ['PTS', 'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV', 'BLKA', 'PF', 'PFD']:
    df_raw[col] = df_raw[col] / df_raw['MIN'] * 36

df_raw.drop(columns='MIN', inplace=True)

### Correlated

In [ ]:
columns_to_drop = [
    'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA',   # Redundant with shooting percentages
    'REB',                                        # Redundant with OREB and DFREB
    'PLUS_MINUS',                                 # Highly dependent on team quality
    'DD2', 'TD3',                                 # Double-dobules and triple-doubles are correlated with other metrics
    'NBA_FANTASY_PTS', 'WNBA_FANTASY_PTS'         # Computed directly from a formula using other stats
]

In [ ]:
df_raw.drop(columns=columns_to_drop, inplace=True)

### Imputation

There are 3 players with no listed weight. We just impute the median weight of players with the same height.

In [ ]:
df_raw['WEIGHT'] = df_raw.groupby('HEIGHT')['WEIGHT'].transform(lambda x: x.fillna(x.median()))

# Experiment design

| Model Type                     | Feature Set A (Skills Only) | Feature Set B (Skills + Bio) |
|--------------------------------|-----------------------------|------------------------------|
| Supervised (Random Forest/SVM) |	Evaluates how well on-court play style predicts a player's official traditional position. |	Evaluates how much physical size dictates a player's official traditional position. |
| Unsupervised (K-Means/GMM)	 | Discovers the "natural" functional roles in the modern NBA based purely on basketball utility. | Discovers clusters dictated primarily by human body types. |

1. The Splits

    df_pure_train (80% of pure players)

    df_pure_test (20% of pure players)

    df_all_clustering = All 100% of df_pure + df_mixed + df_none

2. The Training Phase

    Supervised: Trains only on df_pure_train.

    Unsupervised: Trains blindly on df_all_clustering (asking for exactly 3 clusters).

3. The Final Evaluation & Comparison Phase

    Supervised Evaluation: Test it on df_pure_test to get your baseline classification accuracy (e.g., "Our classifier is 88% accurate at guessing traditional labels based on skill").

    The "Tweener" Analysis: Run your supervised model on df_mixed and df_none. (e.g., "The classifier forces 70% of G-F players into the 'Guard' bucket").

    The Unsupervised Mapping: Look at how the unsupervised model handled the whole league. (e.g., "Our skill-only clustering naturally split the league into Rim Protectors, Floor Spacers, and Playmakers").

    The Ultimate Showdown (The Comparison): Pull the df_pure_test rows from both models. Compare the Supervised Predictions against the Unsupervised Cluster Assignments using the Adjusted Rand Index (ARI) and a Confusion Matrix.

This final step answers your ultimate thesis question: When looking at a completely unbiased holdout set, do the natural, skill-based clusters mathematically resemble the NBA's traditional labels?

You have designed a brilliant pipeline. You are completely ready to drop this into Python, scale your features, and run the code!

# Data split

In [ ]:
df_pure = df_raw[df_raw['POSITION'].isin(['F', 'G', 'C'])].copy().drop(columns="ID")
df_mixed = df_raw[df_raw['POSITION'].isin(['G-F', 'F-G', 'F-C', 'C-F'])].copy().drop(columns="ID")
df_none = df_raw[df_raw['POSITION'].isna()].copy().drop(columns="ID")

In [ ]:
# Build versions without bio columns (keep only stats columns)
dfs_pure = df_pure.drop(columns=['AGE', 'HEIGHT', 'WEIGHT'])
dfs_mixed = df_mixed.drop(columns=['AGE', 'HEIGHT', 'WEIGHT'])
dfs_none = df_none.drop(columns=['AGE', 'HEIGHT', 'WEIGHT'])

In [ ]:
def pure_position_train_test_split(
    df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Isolate features (X) from target (y)
    X = df.drop(columns='POSITION')
    y = df['POSITION']

    # Perform the stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20, 
        random_state=13,  # Ensures reproducible results
        stratify=y        # Stratify to avoid class imbalance
    )

    # Reconstruct df_pure_train df_pure_test dataframes
    df_train = X_train.copy()
    df_train['POSITION'] = y_train

    df_test = X_test.copy()
    df_test['POSITION'] = y_test

    return df_train, df_test

In [ ]:
df_pure_train, df_pure_test = pure_position_train_test_split(df_pure)

In [ ]:
dfs_pure_train = df_pure_train.drop(columns=['AGE', 'HEIGHT', 'WEIGHT'])
dfs_pure_test = df_pure_test.drop(columns=['AGE', 'HEIGHT', 'WEIGHT'])

# Exploratory Data Analysis

In [ ]:
df_pure_train.corr(numeric_only=True)

In [ ]:
def plot_metric_boxplot(df: pd.DataFrame, metric: str) -> None:
    fig = px.box(
        df,
        x="POSITION",
        y=metric,
        title=f"<b>{metric} distribution by Position</b>",
        # labels={"ClaimNb": "Number of Claims", "Exposure": "Policy Exposure (Years)"},
        # points=False # Hides outliers to keep the chart clean and fast
    )

    fig.update_layout(
        showlegend=False,
        template="plotly_dark",
        width=600
    )

    fig.show()

In [ ]:
plot_metric_boxplot(df_pure_train, 'BLK')